In [34]:
# 3-layer MLP (2-3-2) built from scratch
## 2 nodes in the input layer,
## 3 nodes in the hidden layer,
## 2 nodes in the output layer

import numpy as np

In [35]:
# define activation and derivatives
def sigmoid(z):
    """The sigmoid function."""
    return 1.0 / (1.0 + np.exp(-z))


def sigmoid_prime(z):
    """Derivative of the sigmoid function."""
    return sigmoid(z) * (1 - sigmoid(z))

In [36]:
def init_parameters():
    """Parameter initialization.
    W2(0) = [[1,-1],[-1,1],[0.5,-0.5]], W3(0) = [[0.1,-0.1,0.1],[-0.1,0.1,-0.1]]
    b2(0) = [0,0,0], b3(0) = [0,0]
    """
    nodes = [2, 3, 2]
    weights = []
    biases = []
    W2 = np.array([[1.0, -1.0], [-1.0, 1.0], [0.5, -0.5]])
    W3 = np.array([[0.1, -0.1, 0.1], [-0.1, 0.1, -0.1]])
    weights.append(W2)
    weights.append(W3)

    for i in range(1, len(nodes)):
        b = np.zeros((nodes[i], 1))
        biases.append(b)

    return nodes, weights, biases

In [37]:
def feedforward(weights, biases, x):
    """Return the z's and activations of the network if ``x`` is input.
    activations[0] = x (input), activations[1] = a2, activations[2] = a3
    zs[0] = z2, zs[1] = z3
    """
    zs = []
    activations = [x]  # store input as the first activation
    a = x
    for W, b in zip(weights, biases):
        z = W @ a + b
        a = sigmoid(z)
        zs.append(z)
        activations.append(a)

    return zs, activations

In [42]:
## Part c) Testing feedforward using the first training sample x = [2.5, 5]^T
x = np.array([[2.5], [5.0]])
nodes, weights, biases = init_parameters()
zs, activations = feedforward(weights, biases, x)

print("=== Feedforward Test with x = [2.5, 5]^T ===")
print(f"z2 = \n{zs[0]}")
print(f"a2 = \n{activations[1]}")
print(f"z3 = \n{zs[1]}")
print(f"a3 (output) = \n{activations[2]}")

=== Feedforward Test with x = [2.5, 5]^T ===
z2 = 
[[-2.5 ]
 [ 2.5 ]
 [-1.25]]
a2 = 
[[0.07585818]
 [0.92414182]
 [0.22270014]]
z3 = 
[[-0.06255835]
 [ 0.06255835]]
a3 (output) = 
[[0.48436551]
 [0.51563449]]


In [43]:
## Training Data Setup
# Movies I liked: Mary's ratings and John's ratings (in the same order)
mary_liked = [2.5, 3.5, 3.5, 4.5, 4.5]
john_liked = [5, 5, 4, 5, 4]

# Movies I disliked: Mary's ratings and John's ratings (in the same order)
mary_disliked = [1, 1, 1.5, 2.5, 2.5, 2.5]
john_disliked = [5, 4, 4, 3, 1.5, 1]

# Build training data: x = [Mary's rating, John's rating]^T
# Like -> y = [1, 0]^T, Dislike -> y = [0, 1]^T
training_data = []
for m, j in zip(mary_liked, john_liked):
    x = np.array([[m], [j]])
    y = np.array([[1.0], [0.0]])  # like
    training_data.append((x, y))

for m, j in zip(mary_disliked, john_disliked):
    x = np.array([[m], [j]])
    y = np.array([[0.0], [1.0]])  # dislike
    training_data.append((x, y))

print(f"Total training samples: {len(training_data)}")
for i, (x, y) in enumerate(training_data):
    label = "Like" if y[0, 0] == 1 else "Dislike"
    print(f"  Sample {i + 1}: x=[{x[0, 0]}, {x[1, 0]}], label={label}")

Total training samples: 11
  Sample 1: x=[2.5, 5.0], label=Like
  Sample 2: x=[3.5, 5.0], label=Like
  Sample 3: x=[3.5, 4.0], label=Like
  Sample 4: x=[4.5, 5.0], label=Like
  Sample 5: x=[4.5, 4.0], label=Like
  Sample 6: x=[1, 5], label=Dislike
  Sample 7: x=[1, 4], label=Dislike
  Sample 8: x=[1.5, 4.0], label=Dislike
  Sample 9: x=[2.5, 3.0], label=Dislike
  Sample 10: x=[2.5, 1.5], label=Dislike
  Sample 11: x=[2.5, 1.0], label=Dislike


## Part d) Backpropagation and Gradient Calculation

Using MSE cost: $C = \frac{1}{2} \|a^3 - y\|^2$

**Output layer error:**  
$\delta^3 = (a^3 - y) \odot \sigma'(z^3)$

**Hidden layer error:**  
$\delta^2 = (W^3)^T \delta^3 \odot \sigma'(z^2)$

**Gradients:**  
$\nabla_{W^3} C = \delta^3 (a^2)^T$, $\nabla_{b^3} C = \delta^3$  
$\nabla_{W^2} C = \delta^2 \, x^T$, $\nabla_{b^2} C = \delta^2$

In [44]:
def backprop(activations, zs, weights, biases, y):
    """Backpropagation to compute gradients of cost w.r.t. weights and biases.

    Arguments:
        activations: list of activations from feedforward [x, a2, a3]
        zs: list of z values from feedforward [z2, z3]
        weights: list of weight matrices [W2, W3]
        biases: list of bias vectors [b2, b3]
        y: ground truth label (column vector)

    Returns:
        nabla_ws: list of gradients for weights [dC/dW2, dC/dW3]
        nabla_bs: list of gradients for biases [dC/db2, dC/db3]
    """
    nabla_ws = [np.zeros(W.shape) for W in weights]
    nabla_bs = [np.zeros(b.shape) for b in biases]
    num_layers = len(weights) + 1  # total layers including input

    # Output layer error: delta^L = (a^L - y) * sigma'(z^L)
    delta = (activations[-1] - y) * sigmoid_prime(zs[-1])
    nabla_bs[-1] = delta
    nabla_ws[-1] = delta @ activations[-2].T

    # Propagate error backward through hidden layers
    for l in range(2, num_layers):
        z = zs[-l]
        sp = sigmoid_prime(z)
        delta = (weights[-l + 1].T @ delta) * sp
        nabla_bs[-l] = delta
        nabla_ws[-l] = delta @ activations[-l - 1].T

    return nabla_ws, nabla_bs

In [45]:
## Test backprop using the first training sample x = [2.5, 5]^T, y = [1, 0]^T
x = np.array([[2.5], [5.0]])
y = np.array([[1.0], [0.0]])  # like

nodes, weights, biases = init_parameters()
zs, activations = feedforward(weights, biases, x)
nabla_ws, nabla_bs = backprop(activations, zs, weights, biases, y)

print("=== Backprop Test with x = [2.5, 5]^T, y = [1, 0]^T ===")
print(f"nabla_W2 (dC/dW2) = \n{nabla_ws[0]}")
print(f"nabla_b2 (dC/db2) = \n{nabla_bs[0]}")
print(f"nabla_W3 (dC/dW3) = \n{nabla_ws[1]}")
print(f"nabla_b3 (dC/db3) = \n{nabla_bs[1]}")

=== Backprop Test with x = [2.5, 5]^T, y = [1, 0]^T ===
nabla_W2 (dC/dW2) = 
[[-0.00451407 -0.00902814]
 [ 0.00451407  0.00902814]
 [-0.01114644 -0.02229288]]
nabla_b2 (dC/db2) = 
[[-0.00180563]
 [ 0.00180563]
 [-0.00445858]]
nabla_W3 (dC/dW3) = 
[[-0.00976921 -0.11901337 -0.0286799 ]
 [ 0.00976921  0.11901337  0.0286799 ]]
nabla_b3 (dC/db3) = 
[[-0.12878258]
 [ 0.12878258]]


## Part e) Weight Update

Update all weights using gradient descent:  
$W^{(k+1)} = W^{(k)} - \eta \, \frac{\partial C}{\partial W^{(k)}}$

In [46]:
def update(weights, biases, nabla_ws, nabla_bs, eta=0.5):
    """Update weights and biases using gradient descent.

    W_new = W - eta * nabla_W
    b_new = b - eta * nabla_b
    """
    weights = [W - eta * nw for W, nw in zip(weights, nabla_ws)]
    biases = [b - eta * nb for b, nb in zip(biases, nabla_bs)]
    return weights, biases

In [47]:
## Test update using the first training sample x = [2.5, 5]^T
x = np.array([[2.5], [5.0]])
y = np.array([[1.0], [0.0]])  # like
eta = 0.5

nodes, weights, biases = init_parameters()

# Forward pass
zs, activations = feedforward(weights, biases, x)

# Backward pass
nabla_ws, nabla_bs = backprop(activations, zs, weights, biases, y)

# Update
weights, biases = update(weights, biases, nabla_ws, nabla_bs, eta)

print(f"=== Updated Weights & Biases after 1st sample (eta={eta}) ===")
print(f"W2 = \n{weights[0]}")
print(f"W3 = \n{weights[1]}")
print(f"b2 = \n{biases[0]}")
print(f"b3 = \n{biases[1]}")

=== Updated Weights & Biases after 1st sample (eta=0.5) ===
W2 = 
[[ 1.00225703 -0.99548593]
 [-1.00225703  0.99548593]
 [ 0.50557322 -0.48885356]]
W3 = 
[[ 0.10488461 -0.04049332  0.11433995]
 [-0.10488461  0.04049332 -0.11433995]]
b2 = 
[[ 0.00090281]
 [-0.00090281]
 [ 0.00222929]]
b3 = 
[[ 0.06439129]
 [-0.06439129]]


## Part f) Complete One Epoch — Train on All 11 Samples

Repeat forward pass → backprop → update for each of the 11 training samples (online SGD).

In [48]:
## Part f) One complete epoch: forward -> backprop -> update for all 11 samples
eta = 0.5
nodes, weights, biases = init_parameters()

for i, (x, y) in enumerate(training_data):
    # Forward pass
    zs, activations = feedforward(weights, biases, x)
    # Backward pass
    nabla_ws, nabla_bs = backprop(activations, zs, weights, biases, y)
    # Update weights
    weights, biases = update(weights, biases, nabla_ws, nabla_bs, eta)

    label = "Like" if y[0, 0] == 1 else "Dislike"
    output = activations[-1]
    print(
        f"Sample {i + 1}: x=[{x[0, 0]:.1f}, {x[1, 0]:.1f}] ({label}) -> output=[{output[0, 0]:.4f}, {output[1, 0]:.4f}]"
    )

print("\n=== Weights & Biases after 1 epoch ===")
print(f"W2 = \n{weights[0]}")
print(f"W3 = \n{weights[1]}")
print(f"b2 = \n{biases[0]}")
print(f"b3 = \n{biases[1]}")

Sample 1: x=[2.5, 5.0] (Like) -> output=[0.4844, 0.5156]
Sample 2: x=[3.5, 5.0] (Like) -> output=[0.5224, 0.4776]
Sample 3: x=[3.5, 4.0] (Like) -> output=[0.5596, 0.4404]
Sample 4: x=[4.5, 5.0] (Like) -> output=[0.5858, 0.4142]
Sample 5: x=[4.5, 4.0] (Like) -> output=[0.6206, 0.3794]
Sample 6: x=[1.0, 5.0] (Dislike) -> output=[0.5994, 0.4006]
Sample 7: x=[1.0, 4.0] (Dislike) -> output=[0.5686, 0.4314]
Sample 8: x=[1.5, 4.0] (Dislike) -> output=[0.5383, 0.4617]
Sample 9: x=[2.5, 3.0] (Dislike) -> output=[0.5382, 0.4618]
Sample 10: x=[2.5, 1.5] (Dislike) -> output=[0.5387, 0.4613]
Sample 11: x=[2.5, 1.0] (Dislike) -> output=[0.5111, 0.4889]

=== Weights & Biases after 1 epoch ===
W2 = 
[[ 1.01437046 -0.98784948]
 [-0.96492962  1.0238537 ]
 [ 0.51524226 -0.51263783]]
W3 = 
[[ 0.04613197 -0.18425865  0.0519856 ]
 [-0.04613197  0.18425865 -0.0519856 ]]
b2 = 
[[-0.00350135]
 [ 0.0122357 ]
 [-0.01042369]]
b3 = 
[[-0.1331333]
 [ 0.1331333]]


## Part g) Train for 150 Epochs

Repeat the full training process (all 11 samples per epoch) for 150 epochs. Output the resulting weights and biases.

In [51]:
## Part g) Train for 150 epochs
num_epochs = 250
eta = 0.5

# Re-initialize from scratch
nodes, weights, biases = init_parameters()

for epoch in range(num_epochs):
    epoch_cost = 0.0
    for x, y in training_data:
        # Forward pass
        zs, activations = feedforward(weights, biases, x)
        # Accumulate cost for monitoring
        epoch_cost += 0.5 * np.sum((activations[-1] - y) ** 2)
        # Backward pass
        nabla_ws, nabla_bs = backprop(activations, zs, weights, biases, y)
        # Update weights
        weights, biases = update(weights, biases, nabla_ws, nabla_bs, eta)

    if (epoch + 1) % 25 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1:3d}/{num_epochs} | Cost: {epoch_cost:.6f}")

print("\n=== Final Weights & Biases after 150 epochs ===")
print(f"W2 = \n{weights[0]}")
print(f"W3 = \n{weights[1]}")
print(f"b2 = \n{biases[0]}")
print(f"b3 = \n{biases[1]}")

Epoch   1/250 | Cost: 2.816954
Epoch  25/250 | Cost: 2.349018
Epoch  50/250 | Cost: 1.790753
Epoch  75/250 | Cost: 1.473801
Epoch 100/250 | Cost: 1.251574
Epoch 125/250 | Cost: 0.996812
Epoch 150/250 | Cost: 0.885018
Epoch 175/250 | Cost: 0.850056
Epoch 200/250 | Cost: 0.660419
Epoch 225/250 | Cost: 0.429976
Epoch 250/250 | Cost: 0.159590

=== Final Weights & Biases after 150 epochs ===
W2 = 
[[ 1.02851893 -1.97576351]
 [-2.78016743 -0.28372273]
 [ 1.29055735  0.6918761 ]]
W3 = 
[[-3.52967362 -4.30914486  3.6767282 ]
 [ 3.52967362  4.30914486 -3.6767282 ]]
b2 = 
[[ 4.0861113 ]
 [ 7.31762993]
 [-6.35779623]]
b3 = 
[[ 0.39737234]
 [-0.39737234]]


## Part h) Classify "Gravity"

Mary's rating = 3, John's rating = 3.  
Use the trained network to predict: **should I watch Gravity?**

- Output node 1 → "Like"  
- Output node 2 → "Dislike"

In [52]:
## Part h) Classify "Gravity" using the trained network
# Mary's rating = 3, John's rating = 3
x_gravity = np.array([[3.0], [3.0]])

zs, activations = feedforward(weights, biases, x_gravity)
output = activations[-1]

print("=== Classification of 'Gravity' ===")
print(f"Input: Mary={x_gravity[0, 0]}, John={x_gravity[1, 0]}")
print(f"Network output: Like={output[0, 0]:.4f}, Dislike={output[1, 0]:.4f}")
print()
if output[0, 0] > output[1, 0]:
    print("Prediction: LIKE -> Yes, you should watch Gravity!")
else:
    print("Prediction: DISLIKE -> No, you should not watch Gravity.")

# Also test on all training data to see accuracy
print("\n=== Training Data Classification ===")
correct = 0
for i, (x, y) in enumerate(training_data):
    zs_t, acts_t = feedforward(weights, biases, x)
    pred = acts_t[-1]
    pred_label = "Like" if pred[0, 0] > pred[1, 0] else "Dislike"
    true_label = "Like" if y[0, 0] == 1 else "Dislike"
    match = "✓" if pred_label == true_label else "✗"
    if pred_label == true_label:
        correct += 1
    print(
        f"  Sample {i + 1}: x=[{x[0, 0]:.1f}, {x[1, 0]:.1f}] | True: {true_label} | Pred: {pred_label} ({pred[0, 0]:.4f}, {pred[1, 0]:.4f}) {match}"
    )

print(
    f"\nTraining Accuracy: {correct}/{len(training_data)} = {100 * correct / len(training_data):.1f}%"
)

=== Classification of 'Gravity' ===
Input: Mary=3.0, John=3.0
Network output: Like=0.1900, Dislike=0.8100

Prediction: DISLIKE -> No, you should not watch Gravity.

=== Training Data Classification ===
  Sample 1: x=[2.5, 5.0] | True: Like | Pred: Like (0.7830, 0.2170) ✓
  Sample 2: x=[3.5, 5.0] | True: Like | Pred: Like (0.9535, 0.0465) ✓
  Sample 3: x=[3.5, 4.0] | True: Like | Pred: Like (0.7920, 0.2080) ✓
  Sample 4: x=[4.5, 5.0] | True: Like | Pred: Like (0.9543, 0.0457) ✓
  Sample 5: x=[4.5, 4.0] | True: Like | Pred: Like (0.7793, 0.2207) ✓
  Sample 6: x=[1.0, 5.0] | True: Dislike | Pred: Dislike (0.0413, 0.9587) ✓
  Sample 7: x=[1.0, 4.0] | True: Dislike | Pred: Dislike (0.0255, 0.9745) ✓
  Sample 8: x=[1.5, 4.0] | True: Dislike | Pred: Dislike (0.0414, 0.9586) ✓
  Sample 9: x=[2.5, 3.0] | True: Dislike | Pred: Dislike (0.0642, 0.9358) ✓
  Sample 10: x=[2.5, 1.5] | True: Dislike | Pred: Dislike (0.0087, 0.9913) ✓
  Sample 11: x=[2.5, 1.0] | True: Dislike | Pred: Dislike (0.0064, 